# 🧠 Dry Bean Classification with PyTorch

This Jupyter Notebook performs classification on the Dry Bean dataset using a feedforward neural network implemented in PyTorch.

We will go through the following steps:

1. Load and inspect the dataset
2. Exploratory Data Analysis (EDA)
3. Preprocessing (encoding, scaling, splitting)
4. Define and train a PyTorch model
5. Evaluate the model
6. Interpret the results


## 📥 Step 1: Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Load the dataset
df = pd.read_csv("Lecture_10_Dry_Bean_Dataset.csv")
df.head()

### 🔍 Data Info and Summary

In [ ]:
print(df.info())
print(df.describe())
print(df.isnull().sum())

## 📊 Step 2: Exploratory Data Analysis (EDA)

In [ ]:
# Visualize class distribution
sns.countplot(x='Class', data=df)
plt.xticks(rotation=45)
plt.title('Class Distribution')
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(12, 8))
sns.heatmap(df.drop('Class', axis=1).corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

## 🧹 Step 3: Preprocessing

In [ ]:
# Encode class labels
y = df['Class']
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
X = df.drop("Class", axis=1)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

### Convert to PyTorch Tensors

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

## 🧠 Step 4: Define the Neural Network

In [ ]:
class BeanClassifier(nn.Module):
    def __init__(self):
        super(BeanClassifier, self).__init__()
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 7)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

## 🏋️ Step 5: Training the Model

In [ ]:
model = BeanClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

## 📈 Step 6: Evaluation and Interpretation

In [ ]:
model.eval()
y_pred = []
y_true = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        y_pred.extend(predicted.numpy())
        y_true.extend(y_batch.numpy())

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))


### 📌 Step 5: Training Interpretation

- The model used **ReLU activation** to allow non-linear transformations.
- **Adam optimizer** adapts learning rates during training for better convergence.
- The decreasing training loss across 20 epochs suggests the model successfully learned patterns in the training data.
- **CrossEntropyLoss** was used to measure the classification error and guide weight updates.


In [ ]:
# Confusion matrix
conf_mat = confusion_matrix(y_true, y_pred)
sns.heatmap(conf_mat, annot=True, fmt='d',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

✅ **Interpretation**:
- Precision, recall, and F1-scores help us understand per-class performance.
- A good model will have a strong diagonal in the confusion matrix (indicating correct predictions).
- Misclassifications may suggest overlapping features or imbalance.


## 📘 Theoretical Background from Lectures

This project applies key concepts from the lecture slides:

---

### 🔹 Perceptron & Feedforward Networks (Lecture 09)
- As introduced in Lecture 09, a **perceptron** is the basic unit of a neural network: it combines weighted inputs, adds a bias, and applies an activation function.
- We use **ReLU** as the activation function, one of the most popular due to its performance and simplicity (Slide 13).
- The final layer outputs raw values ("logits") suitable for classification with **CrossEntropyLoss** (Slide 21).

---

### 🔹 Gradient Descent and Backpropagation
- Training the network involves minimizing a loss function via **gradient descent** (Slides 24–29).
- PyTorch handles **backpropagation** automatically, updating weights to reduce classification error.

---

### 🔹 Architecture Design (Lecture 10)
- Instead of using a single large layer, we split the model into multiple hidden layers for better learning efficiency (Slide 7, Lecture 10).
- The architecture `16 → 64 → 32 → 7` balances complexity and computational cost.

---

### 🔹 Overfitting & Generalization
- To reduce overfitting, techniques such as **dropout** or **early stopping** (Slide 35) could be integrated.
- Although not included in this version, these are valuable for larger models or datasets with imbalance/noise.

---

### 🔹 Interpretation
- A strong diagonal in the confusion matrix indicates the model is learning correct class mappings.
- **Precision, recall, and F1-score** highlight where the model performs well and where improvements could be made.



### 📌 Step 6: Evaluation Interpretation

#### 🔹 **Performance Summary**:
- **BOMBAY** was classified with perfect precision (1.00) and nearly perfect recall (0.99).
- **CALI, HOROZ, SEKER** also achieved high f1-scores (above 0.93), indicating strong performance.
- **SIRA** had the lowest scores: precision (0.88) and recall (0.83), suggesting it was more difficult to classify.

#### 🔹 **Confusion Matrix Insights**:
- The **strong diagonal** in the matrix confirms correct predictions.
- Major misclassifications:
  - **SIRA → DERMASON** (71 times)
  - **BARBUNYA → CALI** (22 times)

These results indicate some feature overlap and challenge in distinguishing similar classes.
